In [8]:
# !pip install -q unsloth
# !pip install -q datasets

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*use_return_dict.*')
warnings.filterwarnings('ignore', message='.*max_new_tokens.*max_length.*')
warnings.filterwarnings("ignore", category=UserWarning, message=".*IProgress not found.*")

In [1]:
import torch, random

print(f"PyTorch:  {torch.__version__}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch:  2.10.0+cu128
GPU:      NVIDIA GeForce RTX 3070
VRAM:     8.6 GB


### Base Model

In [9]:
from unsloth import FastLanguageModel

BASE_MODEL="HuggingFaceTB/SmolLM-135M"
MAX_SEQ_LENGTH=1024
SEED=42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print(f"[OK] Model loaded: {BASE_MODEL}")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 330.95it/s]


[OK] Model loaded: HuggingFaceTB/SmolLM-135M


In [10]:
def generate_text(model, tokenizer, prompt, max_new_tokens=128):
  """ Generates a completions for a given prompt """
  inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
  with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True, # KV cache
        do_sample=True,
        repetition_penalty=1.2,
        temperature=0.7,
        top_p=0.9,
    )
  new_tokens = outputs[0, inputs.input_ids.shape[1]:]
  return tokenizer.decode(new_tokens, skip_special_tokens=True)

### Load CPT model trained on SEC 10K filings

In [11]:
from peft import PeftModel

# load LoRA adapter
model = PeftModel.from_pretrained(model, "./cpt_sec_adapter")

# since from last session we know that "SmolLM-135M" dosen't perform good on financial data,
# so we load our finetuned model which has financial domain knowledge

/home/junaid/miniconda3/envs/ml/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


### Testing Model's answering ablites

In [12]:
# The Aplaca-style template for SFT use
ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [13]:
# Test Give the base model an Aplaca-formated prompt
test_questions = [
    ("What was the company's total revenue?", "The company reported total revenues of $53.8 billion for the fiscal year ended September 30, 2023, compared to $52.1 billion for the prior year."),
    ("What are the main risk factors mentioned?", "Risk factors include fluctuations in foreign currency exchange rates, changes in government regulations, and increased competition in key markets."),
    ("What is EBITDA and why is it important?",""),
]

print("="*60)
print("BASE MODEL -- Attempting Q&A (before SFT)")
print("="*60)
for question, context in test_questions:
  prompt = ALPACA_PROMPT.format(question, context, "")  # empty response
  completion = generate_text(model, tokenizer, prompt)
  print(f"\nQ: {question}")
  print(f"\nC: {context}")
  print(f"\nA: {completion}")
  print("-"*60)

Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL -- Attempting Q&A (before SFT)


Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What was the company's total revenue?

C: The company reported total revenues of $53.8 billion for the fiscal year ended September 30, 2023, compared to $52.1 billion for the prior year.

A: Given below are statements about the Company and its financial conditions in the following sections which describe what will be presented at each end date or period as part of this discussion (as defined by the information provided): (i) The Company does not expect any future loss on assets sold during the periods covered under this section because it has been unable to obtain cash from these transactions. (ii) The Company anticipates continuing growth over time through continued sales expansion; however, such expected increases may include significant losses incurred due to unforeseen circumstances - including changes in management practices related to operations etc., resulting either directly or indirectly from increasing costs associated with increased volume production ,
------------------

Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What are the main risk factors mentioned?

C: Risk factors include fluctuations in foreign currency exchange rates, changes in government regulations, and increased competition in key markets.

A: The following information has been omitted from this portion of the report for lack of time or space to complete it satisfactorily by the employee who wrote it (not shown): "There were approximately 136 persons employed at the Company's retail locations during the quarter ended March 28, 1994." This number was estimated as follows based on prior years' experience when estimating numbers available for review/review : Number Employees Established ; During December through June , The company had 50 employees established . At January - end ? Year = = = Annual Reports = = = = = = = = = = = = =
------------------------------------------------------------

Q: What is EBITDA and why is it important?

C: 

A: Earnings = 1302784 $(5) Income Taxes =$96423 (not shown in this report as of December 31,

**Observation:** Even though model sounds like SEC 10k filing buy the model doesn't understand the template. It autocompletes randomly -- it doesn't know that "### Response:" means answer the question here.

### Prepate SFT Dataset (virattt/financial-qa-10k)


In [14]:
from datasets import load_dataset

dataset = load_dataset("virattt/financial-qa-10k", split="train")

print(f"Dataset size: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")
print(f"\nSample")
print(dataset[0])

Generating train split: 100%|██████████| 7000/7000 [00:00<00:00, 154747.71 examples/s]

Dataset size: 7000
Dataset columns: ['question', 'answer', 'context', 'ticker', 'filing']

Sample
{'question': 'What area did NVIDIA initially focus on before expanding to other computationally intensive fields?', 'answer': 'NVIDIA initially focused on PC graphics.', 'context': 'Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.', 'ticker': 'NVDA', 'filing': '2023_10K'}


In [15]:
# Lets look at a few examples
for i in range(3):
  ex = dataset[i]
  print(f"\n--- Example {i} ---")
  print(f"Q: {ex['question']}")
  print(f"C: {ex['context']}")
  print(f"A: {ex['answer']}")


--- Example 0 ---
Q: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?
C: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.
A: NVIDIA initially focused on PC graphics.

--- Example 1 ---
Q: What are some of the recent applications of GPU-powered deep learning as mentioned by NVIDIA?
C: Some of the most recent applications of GPU-powered deep learning include recommendation systems, which are AI algorithms trained to understand the preferences, previous decisions, and characteristics of people and products using data gathered about their interactions, large language models, which can recognize, summarize, translate, predict and generate text and other content based on knowledge gained from massive datasets, and generative AI, which uses algorithms that create new content, including audio, code, images, text, simulations, and videos, based on the data they hav

In [37]:
# Convert Dataset to Alpaca-format

EOS_TOKEN = tokenizer.eos_token
print(f"EOS token: '{EOS_TOKEN}' (id: {tokenizer.eos_token_id})")

def format_to_alpaca(examples):
  """Convert dataset rows to Alpaca-formated training text."""
  instructions = examples.get("question")
  inputs = examples.get("context")
  answers = examples.get("answer")

  alpaca_formated_texts = []
  for inst, inputs, ans in zip(instructions, inputs, answers):
    text = ALPACA_PROMPT.format(inst, inputs, ans) + EOS_TOKEN
    alpaca_formated_texts.append(text)

  return {"text": alpaca_formated_texts}

# Apply formating
formatted_dataset = dataset.map(
    format_to_alpaca,
    batched=True,
    remove_columns=dataset.column_names,
)

print(f"\nFormatted dataset: {len(formatted_dataset)} examples")
print("="*60)
print(f"Sample formatted text (first 400 chars):")
print(formatted_dataset[0]["text"][:500])
print("...")

EOS token: '<|endoftext|>' (id: 0)

Formatted dataset: 7000 examples
Sample formatted text (first 400 chars):
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What area did NVIDIA initially focus on before expanding to other computationally intensive fields?

### Input:
Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

### Response:
NVIDIA initially focused on PC graphics.<|endoftext|>
...


In [38]:
# Test/Train split
split = formatted_dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

Train dataset size: 6650
Eval dataset size: 350


### SFT with Unsloth and LoRA

Key differences from CPT:
- **Rank 16** (not 32) -- less capacity needed for behavior change
- **No embed_tokens/lm_head** -- vocabulary is fine, behavior needs to change
- **Packing still on** -- efficient use of GPU
- **EOS in every example** -- teaches the model to stop

In [39]:
# Free model and reload fresh
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False  # full precision for merging
)

# Attach and merge CPT adapter
model = PeftModel.from_pretrained(model, "./cpt_sec_adapter")
model = model.merge_and_unload()

# Save meged model to disk
model.save_pretrained("./cpt_merged")
tokenizer.save_pretrained("./cpt_merged")
print("[OK] Model meged and saved")


# Free model and reload fresh
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./cpt_merged",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True  # safe now, fresh load
)
print("[OK] Reloaded merged model in 4bit")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 675.88it/s]
/home/junaid/miniconda3/envs/ml/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/home/junaid/miniconda3/envs/ml/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:629: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications. You can opt to merge the adapter after cloning the weights (to untie the embeddings). You can untie the embeddings by loading the model with `tie_word_embeddings=False`. For example:
```python
from transformers import AutoModelForCau

[OK] Model meged and saved
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!



Loading weights: 100%|██████████| 273/273 [00:00<00:00, 691.90it/s]


[OK] Reloaded merged model in 4bit


In [40]:
# Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj",
        # No embed_tokens and lm_head as SFT doesnot learn new vocablary
        # we only need to change the behaviour
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters:     {total_params:>12,}")
print(f"Trainable (LoRA):     {trainable_params:>12,}  ({100*trainable_params/total_params:.2f}%)")
print(f"Frozen (base model):  {total_params - trainable_params:>12,}")


Total parameters:      114,626,880
Trainable (LoRA):        4,884,480  (4.26%)
Frozen (base model):   109,742,400


In [ ]:
from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

training_args = UnslothTrainingArguments(
    output_dir="./sft_financial_qa_training_args",

    # --- Core ---
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,     # Effective batch = 8*4 = 32

    # --- Learning rate ---
    learning_rate=2e-4,
    # No embedding_learning_rate needed -- no embed_tokens in LoRA
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # --- Optimization ---
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # --- Data ---
    max_length=MAX_SEQ_LENGTH,
    packing=True,                     # Pack short Q&A pairs together
    dataset_text_field="text",
    dataset_num_proc=2,

    # --- Eval ---
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=8,

    # --- Checkpointing ---
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- Logging ---
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

print("[OK] Trainer configured")
print(f"Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 350/350 [00:00<00:00, 386.26 examples/s]

[OK] Trainer configured
Train: 6,650 | Eval: 350


In [42]:
# TRAIN!
# ~4-6 minutes on RTX 3070 with 135M model + 7K examples + 2 epochs

trainer_stats = trainer.train()
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,650 | Num Epochs = 2 | Total steps = 416
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 4,884,480 of 167,711,040 (2.91% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,1.573932,1.478050
100,1.347600,1.363869
150,1.346383,1.331604
200,1.307824,1.311478
250,1.251586,1.299107
300,1.250919,1.291733
350,1.261614,1.288640
400,1.296107,1.287807
416,1.287182,1.287822


Final train loss: 1.4160


### Moment of Truth

In [46]:
FastLanguageModel.for_inference(model)

print("="*70)
print("After SFT --- Financial Q&A")
print("="*70)
for question, context in test_questions:
  # Give it the template with empty res and let it fill in
  prompt = ALPACA_PROMPT.format(question, context, "")
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=150)

  print(f"\nQ: {question}")
  print(f"C: {context}")
  print(f"A: {completion}")
  print("-"*50)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


After SFT --- Financial Q&A


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What was the company's total revenue?
C: The company reported total revenues of $53.8 billion for the fiscal year ended September 30, 2023, compared to $52.1 billion for the prior year.
A: $53.8 Billion
--------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What are the main risk factors mentioned?
C: Risk factors include fluctuations in foreign currency exchange rates, changes in government regulations, and increased competition in key markets.
A: Foreign Currency Exchange Rates
--------------------------------------------------

Q: What is EBITDA and why is it important?
C: 
A: Ebitda measures financial results in terms of earnings per share (EPS), which includes net income as well as other metrics such as cash flow from operations or operating leverage on assets compared to debt levels for various industries including manufacturing businesses due to differences among companies. It also considers factors like depreciation expense vs. amortization over time based on different economic conditions affecting inventory costs associated with these items. For instance, if product prices decrease significantly during downtrends caused by higher commodity values but remain relatively stable throughout periods when production may be expected 

### What did model actually learn?

In [ ]:
# Test 1: Out of domain -- non-financial questions
print("=" * 70)
print("TEST: Out-of-domain questions")
print("=" * 70)

ood_questions = [
    ("What is the capital of France?", ""),
    ("Write a poem about the ocean.", ""),
    ("How do I make pasta?", ""),
]

for q, c in ood_questions:
    prompt = ALPACA_PROMPT.format(q, c, "")
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=100)
    print(f"\nQ: {q}")
    print(f"A: {completion}")
    print("-" * 50)

print("\nNote: The model was SFT'd on financial Q&A only.")
print("It learned the BEHAVIOR (answer questions) but may struggle")
print("with non-financial KNOWLEDGE (it's still SmolLM-135M underneath).")

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST: Out-of-domain questions


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the capital of France?
A: France's net real estate assets are $60 billion and its equity equivalent reserves total approximately 538 million acres (149 square miles). The French government owns about half as much as it does in total property sales revenue during fiscal year e
--------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Write a poem about the ocean.
A: The ocean's water and life are our best hope for survival on this planet!
--------------------------------------------------

Q: How do I make pasta?
A: Porridge or water can be used to form long-term storage of short-duration products such as pasta and noodles.|

##### What are some examples from our industry where we use pasta in manufacturing processes for food packaging applications like pizza bo
--------------------------------------------------

Note: The model was SFT'd on financial Q&A only.
It learned the BEHAVIOR (answer questions) but may struggle
with non-financial KNOWLEDGE (it's still SmolLM-135M underneath).


### Evaulating SFT Quality

For SFT, perplexity alone isn't enough. We check:
- Does the model answer the actual question?
- Does it stop generating at the right time?
- Does it hallucinate or make things up?

In [43]:
# Check: does the model actually stop generating?
print("=" * 70)
print("TEST: Does the model stop? (EOS behavior)")
print("=" * 70)

prompt = ALPACA_PROMPT.format(
    "What was net income?",
    "Net income for the fiscal year was $2.3 billion.",
    "",
)

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
with torch.no_grad():
  outputs = model.generate(
      input_ids=inputs.input_ids,
      attention_mask=inputs.attention_mask,
      max_new_tokens=300, # Allow lots of room
      use_cache=True,
      repetition_penalty=1.2,
      temperature=0.7,
  )

new_tokens = outputs[0, inputs.input_ids.shape[1]:]
full_output = tokenizer.decode(new_tokens, skip_special_tokens=False)

print(f"Raw output (with special tokens visible):")
print(repr(full_output[:300]))
print(f"\nTokens generated: {len(new_tokens)}")
print(f"EOS token found: {tokenizer.eos_token_id in new_tokens.tolist()}")

if tokenizer.eos_token_id in new_tokens.tolist():
  eos_pos = new_tokens.tolist().index(tokenizer.eos_token_id)
  print(f"EOS at position: {eos_pos} out of {len(new_tokens)} tokens")
  print("[PASS] Model learned when to stop!")
else:
  print("[WARN] Model didn't produce EOS -- may need more training")

Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST: Does the model stop? (EOS behavior)
Raw output (with special tokens visible):
'$2.3 billion<|endoftext|>'

Tokens generated: 6
EOS token found: True
EOS at position: 5 out of 6 tokens
[PASS] Model learned when to stop!


### Save and Merge the adapter

In [47]:
import os

# Save LoRA adapter (small file)
ADAPTER_PATH = "./sft_financial_qa_adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH) if f.endswith(('.safetensors', '.bin'))
)
print(f"[OK] Adapter saved to {ADAPTER_PATH}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"\nFiles saved:")
for f in sorted(os.listdir(ADAPTER_PATH)):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, f))
    print(f"  {f:<35} {size/1e6:.2f} MB")

[OK] Adapter saved to ./sft_financial_qa_adapter
Adapter size: 19.6 MB

Files saved:
  README.md                           0.01 MB
  adapter_config.json                 0.00 MB
  adapter_model.safetensors           19.59 MB
  tokenizer.json                      3.52 MB
  tokenizer_config.json               0.00 MB


In [48]:
# Merge adapter into base model
# This collapses W + (alpha/r)*BA into a single W_new
# Result: single model, zero inference overhead

MERGED_PATH = "./sft_financial_qa_merged"

# Save merged model in float16
model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method="merged_16bit",
)

merged_size = sum(
    os.path.getsize(os.path.join(MERGED_PATH, f))
    for f in os.listdir(MERGED_PATH) if f.endswith(('.safetensors', '.bin'))
)
print(f"\n[OK] Merged model saved to {MERGED_PATH}")
print(f"Merged model size: {merged_size / 1e6:.1f} MB")
print(f"\nAdapter was {adapter_size/1e6:.1f} MB")
print(f"Merged is {merged_size/1e6:.1f} MB (full model with changes baked in)")
print(f"\nThe merged model runs at the SAME speed as the original.")
print(f"No LoRA overhead. W_new = W + (alpha/r) * B * A. Just one matrix.")

Detected local model directory: /home/junaid/fine_tuning/cpt_merged
Found HuggingFace hub cache directory: /home/junaid/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]


Unsloth: Merge process complete. Saved to `/home/junaid/fine_tuning/sft_financial_qa_merged`

[OK] Merged model saved to ./sft_financial_qa_merged
Merged model size: 325.7 MB

Adapter was 19.6 MB
Merged is 325.7 MB (full model with changes baked in)

The merged model runs at the SAME speed as the original.
No LoRA overhead. W_new = W + (alpha/r) * B * A. Just one matrix.
